# Revised Linear Model

Implementing our Logistic Regression with scikit-learn methods, rather than manually

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pylab as plt
%matplotlib inline
from data_cleaning import cleaned_data

In [10]:
class Dataset:
    """
    Prepare the dataset into training, validation, and test sets.
    """
    def __init__(self, random_state = 123):
        self.data = cleaned_data()
        y_df = self.data["VOTED"].map({"voted": 0, "not_voted": 1}).to_numpy()
        X_df = self.data.drop(columns=["VOTED"])

        categorical_feature = ["SEX", "RACE", "EDUC", "EMPSTAT", "NATIVITY", "REGION",
                               "METRO", "MARST", "DIFFMOB"]
        numerical_feature = ["AGE", "FAMSIZE", "NCHILD", "FAMINC", "INCOME_PER_PERSON"]

        ## training set 70%, validation set 10%, test set 20%
        #added stratify in addition to neural networks code
        train_x, rest_x, self.train_y, rest_y = train_test_split(
            X_df, y_df, test_size=0.3, random_state=random_state, stratify=y_df)
        val_x, test_x, self.val_y, self.test_y = train_test_split(
            rest_x, rest_y, test_size=(2/3), random_state=random_state, stratify=rest_y)
        
        self.ohe = OneHotEncoder(sparse_output=False)
        self.scaler = StandardScaler()

        train_x_categorical = self.ohe.fit_transform(train_x[categorical_feature])
        train_x_numerical = self.scaler.fit_transform(train_x[numerical_feature])
        val_x_categorical = self.ohe.transform(val_x[categorical_feature])
        val_x_numerical = self.scaler.transform(val_x[numerical_feature])
        test_x_categorical = self.ohe.transform(test_x[categorical_feature])
        test_x_numerical = self.scaler.transform(test_x[numerical_feature])

        self.train_x = np.concatenate([train_x_categorical, train_x_numerical], axis=1)
        self.val_x = np.concatenate([val_x_categorical, val_x_numerical], axis=1)
        self.test_x = np.concatenate([test_x_categorical, test_x_numerical], axis=1)

        categorical_feature_name = self.ohe.get_feature_names_out(categorical_feature)
        numerical_feature_name = self.scaler.get_feature_names_out(numerical_feature)
        self.feature_order = np.concatenate([categorical_feature_name, numerical_feature_name])

In [12]:
# Check if the arrays are created correctly.
data = Dataset(random_state=1234)

print(data.train_x.shape)
print(data.train_x[:5])
print(data.feature_order)

(43628, 31)
[[ 0.          1.          0.          0.          0.          0.
   1.          0.          1.          0.          1.          0.
   0.          0.          0.          1.          1.          0.
   0.          0.          1.          0.          0.          1.
   0.          1.          0.09805191 -1.1218798  -0.58658152 -0.88609804
   0.19809166]
 [ 0.          1.          0.          0.          0.          0.
   1.          0.          1.          0.          0.          1.
   0.          0.          0.          1.          0.          0.
   0.          1.          1.          0.          0.          1.
   0.          1.          0.09805191  0.2424131   0.39536072 -1.14727784
  -1.02910957]
 [ 0.          1.          0.          0.          0.          0.
   1.          1.          0.          0.          0.          0.
   1.          0.          0.          1.          0.          0.
   0.          1.          1.          0.          1.          0.
   0.          1. 

Logistic regression with regularization

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate 

def logreg_variations(X, y, penalty = 'l2', C = 1.0, cv = 5, scoring = 'f1'):
    logreg = LogisticRegression(penalty = penalty, C = C, solver = 'liblinear')
    scores = cross_validate(logreg, X, y, cv=cv, return_train_score=True, scoring = scoring)

    return scores

In [ ]:
# get the scores for L1 and L2
l1_version = logreg_variations(data.train_x, data.train_y, peanlty = 'l1')
l2_version = logreg_variations(data.train_x, data.train_y, penalty = 'l2')

# calculate the mean
l1_mean_train_score = np.mean(l1_version["train_score"])
l1_mean_test_score =  np.mean(l1_version["test_score"])

l2_mean_train_score = np.mean(l2_version["train_score"])
l2_mean_test_score =  np.mean(l2_version["test_score"])

print(f'For L1 the mean cross validation train score is {l1_mean_train_score} and the mean test score is {l1_mean_train_score}')
print(f'For L2 the mean cross validation train score is {l2_mean_train_score} and the mean test score is {l2_mean_train_score}')

For L1 the mean cross validation train score is 0.7787800001301244 and the mean test score is 0.7787800001301244
For L2 the mean cross validation train score is 0.7685164758548735 and the mean test score is 0.7685164758548735


Logistic regression with different types of scores

In [ ]:
# Accuracy provides the highest model score

scorers_list = ['accuracy', 'precision','f1', 'recall']

for scorer in scorers_list:
    scores = logreg_variations(data.train_x, data.train_y, penalty = 'l2', C = 0.1, scoring = scorer)
    mean_train_score = np.mean(scores["train_score"])
    mean_test_score =  np.mean(scores["test_score"])
    print(f'With {scorer} the cross validation mean train score is {mean_train_score} and the mean test score is {mean_test_score}')

With accuracy the cross validation mean train score is 0.7715287447781978 and the mean test score is 0.771135866321862
With precision the cross validation mean train score is 0.5397508966066886 and the mean test score is 0.5366000672059654
With f1 the cross validation mean train score is 0.219116945493134 and the mean test score is 0.21773192601325828
With recall the cross validation mean train score is 0.1374860518742827 and the mean test score is 0.13665296142398684


In [ ]:
C_list = [1.0, 0.1, 0.01, 0.001, 0.0001]

for C in C_list:
    logreg = LogisticRegression(l1_ratio = 0, C = C, solver = 'liblinear')
    logreg.fit(data.train_x, data.train_y)
    # organise coefficients and features into a table
    coef_table = pd.DataFrame(zip(data.feature_order, np.transpose(logreg.coef_)), columns=['features', 'coef'])
    print("C:", C)
    print(coef_table)

Cross validation of all hyperparameters: L1/L2, C, scoring

In [18]:
import itertools

l1_ratios = ['l1', 'l2']
C_list = list(np.logspace(-4, 1, 50))  # 0.0001 to 10, 50 points
scorers_list = ['f1', 'precision', 'recall']

hyperparam_combos = list(itertools.product(l1_ratios, C_list, scorers_list))

# create a dictionary to track the scores for different combinations
scores = {}

# TODO: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
for penalty, C, scorer in hyperparam_combos:
    # validate the model with the different combinations of eta and lambda
    all_scores = logreg_variations(data.train_x, data.train_y, penalty = penalty, C = C, scoring = scorer)
    mean_train_score = np.mean(all_scores["train_score"])
    mean_test_score =  np.mean(all_scores["test_score"])

    scores[(penalty, C, scorer)] = (mean_train_score, mean_test_score)

(best_penalty, best_C, best_scorer), (mean_train, mean_test) = max(scores.items(), key=lambda item: item[1][1])
print(f'Combination with the highest score: [penalty = {best_penalty}, C = {best_C}, and scoring = {best_scorer}] \n \t mean train score: {mean_train} \n \t mean test score: {mean_test}')

/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/sit

Combination with the highest score: [penalty = l2, C = 0.00012648552168552957, and scoring = precision] 
 	 mean train score: 0.7491015669558657 
 	 mean test score: 0.7572437256981266


/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Evaluation with a class imbalance

In [26]:
# count number of not voted / voted entries
not_voted = y.sum()
voted = len(data) - not_voted

# get the proportion
prop_not_voted = (not_voted / len(data)) * 100
prop_voted = (voted / len(data)) * 100
print(f'{prop_not_voted}% of our observations did not vote, and {prop_voted}% did vote')

23.37757407937872% of our observations did not vote, and 76.62242592062128% did vote
